# UAE IA Compliance Mapping Pipeline

Works on **Google Colab** and **Lightning.ai** — environment is auto-detected.

| | Google Colab | Lightning.ai |
|---|---|---|
| Storage | Ephemeral — needs Drive | **Persistent** — files survive restarts |
| GPU | T4 free tier | A10G / T4 depending on plan |
| Model save | Copies to Google Drive | Stays in repo dir automatically |
| First run | ~25–40 min | ~25–40 min |
| Re-runs | Restore from Drive (~30s) | Already on disk (~0s) |

**Lightning.ai setup (one time):**
1. Go to [lightning.ai](https://lightning.ai) → New Studio → Jupyter Notebook
2. Select GPU machine (A10G or T4)
3. Open this notebook and run all cells top to bottom

**Colab setup:** `Runtime → Change runtime type → T4 GPU`

## Step 1 — Check GPU

In [2]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('Running on CPU — will be slower (~3-4 hrs). Consider enabling GPU.')

GPU available: True
GPU: Tesla T4
VRAM: 15.8 GB


## Step 2 — Clone the repo

In [3]:
import os, sys
from getpass import getpass

REPO_OWNER = 'puneetha08nr'
REPO_NAME  = 'regulatory_parsing_improved'

# ── Auto-detect environment ───────────────────────────────────────────────
# Lightning.ai: persistent storage lives under /teamspace/studios/this_studio
# Colab:        ephemeral /content (wiped on restart)
IS_LIGHTNING = os.path.isdir('/teamspace')
IS_COLAB     = 'google.colab' in sys.modules or os.path.isdir('/content')

if IS_LIGHTNING:
    BASE_DIR = '/teamspace/studios/this_studio'
    print('Environment: Lightning.ai  (persistent storage — no Drive needed)')
else:
    BASE_DIR = '/content'
    print('Environment: Google Colab  (ephemeral — Drive will be used for persistence)')

REPO_DIR = os.path.join(BASE_DIR, REPO_NAME)

# ── Authentication ────────────────────────────────────────────────────────
TOKEN = getpass('GitHub token (press Enter if repo is public): ').strip()
if TOKEN:
    REPO_URL = f'https://{TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
    print('Using token authentication.')
else:
    REPO_URL = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
    print('No token — assuming public repo.')

# ── Clone or pull ─────────────────────────────────────────────────────────
if not os.path.exists(REPO_DIR):
    print(f'Cloning {REPO_NAME}...')
    ret = os.system(f'git clone "{REPO_URL}" "{REPO_DIR}"')
    if ret != 0:
        raise RuntimeError(
            'git clone failed.\n'
            '  • If the repo is private, re-run this cell and enter a GitHub token.\n'
            '  • Create a token at: GitHub → Settings → Developer settings → '
            'Personal access tokens → Generate new token (repo scope).'
        )
else:
    print('Repo already cloned — pulling latest changes')
    os.system(f'git -C "{REPO_DIR}" pull')
    

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
print(f'REPO_DIR = {REPO_DIR}')

Environment: Lightning.ai  (persistent storage — no Drive needed)
No token — assuming public repo.
Repo already cloned — pulling latest changes
Already up to date.
Working directory: /teamspace/studios/this_studio/regulatory_parsing_improved
REPO_DIR = /teamspace/studios/this_studio/regulatory_parsing_improved


## Step 3 — Copy data files

**Lightning.ai:** Upload your data files once using the file browser (left sidebar).  
Place them in `/teamspace/studios/this_studio/compliance_pipeline_data/` — they persist forever.

**Google Colab:** Upload to Google Drive under `compliance_pipeline_data/` as before.

| File | Destination in repo |
|------|---------------------|
| `policies/` folder | `data/02_processed/policies/` |
| `uae_ia_controls_structured.json` | `data/02_processed/` |
| `golden_mapping_dataset.json` | `data/07_golden_mapping/` |
| `not_applicable_passages.json` | `data/07_golden_mapping/` |
| `obligation-classifier-legalbert/` | repo root |

In [4]:
import shutil, pathlib, subprocess, os

# ── Resolve data source based on environment ──────────────────────────────
if IS_LIGHTNING:
    # Lightning.ai: persistent storage — no Drive mount needed
    DRIVE_DATA = '/teamspace/studios/this_studio/compliance_pipeline_data'
    print('Lightning.ai: reading from persistent storage at', DRIVE_DATA)
else:
    # Colab: mount Drive first
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = '/content/drive/MyDrive/compliance_pipeline_data'
    print('Colab: reading from Google Drive at', DRIVE_DATA)


def copy_from_drive(src, dst, required=False):
    src, dst = pathlib.Path(src), pathlib.Path(dst)
    if not src.exists():
        msg = f'  {"❌ MISSING (required)" if required else "SKIP (optional, not found in Drive)"}: {src.name}'
        print(msg)
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print(f'  ✅ {src.name}')
    return True

print('Copying data from Google Drive...')

# Required: policy passages
copy_from_drive(f'{DRIVE_DATA}/policies',
                f'{REPO_DIR}/data/02_processed/policies', required=True)

# Required: UAE IA controls
copy_from_drive(f'{DRIVE_DATA}/uae_ia_controls_structured.json',
                f'{REPO_DIR}/data/02_processed/uae_ia_controls_structured.json', required=True)

# Optional but recommended: golden dataset (blocklist)
copy_from_drive(f'{DRIVE_DATA}/golden_mapping_dataset.json',
                f'{REPO_DIR}/data/07_golden_mapping/golden_mapping_dataset.json')
copy_from_drive(f'{DRIVE_DATA}/not_applicable_passages.json',
                f'{REPO_DIR}/data/07_golden_mapping/not_applicable_passages.json')

# Optional: LegalBERT model — accepts either a folder OR a zip file
model_dst = pathlib.Path(f'{REPO_DIR}/obligation-classifier-legalbert')
model_folder = pathlib.Path(f'{DRIVE_DATA}/obligation-classifier-legalbert')
model_zip    = pathlib.Path(f'{DRIVE_DATA}/obligation-classifier-legalbert.zip')

if model_folder.exists():
    copy_from_drive(str(model_folder), str(model_dst))
elif model_zip.exists():
    print(f'  Unzipping obligation-classifier-legalbert.zip ...')
    import zipfile
    try:
        with zipfile.ZipFile(str(model_zip), 'r') as zf:
            # Show top-level entries so we know the zip structure
            names = zf.namelist()
            print(f'    zip contains {len(names)} files, e.g.: {names[:3]}')
            zf.extractall(REPO_DIR)
        # The zip may extract to a differently-named folder; rename if needed
        extracted = [pathlib.Path(REPO_DIR) / n.split('/')[0] for n in names if n.strip()]
        extracted_root = extracted[0] if extracted else None
        if extracted_root and extracted_root.exists() and extracted_root != model_dst:
            extracted_root.rename(model_dst)
            print(f'    renamed {extracted_root.name} → obligation-classifier-legalbert')
        print('  ✅ obligation-classifier-legalbert (from zip)')
    except zipfile.BadZipFile as e:
        print(f'  ❌ Bad zip file: {e}')
        print('     Re-zip the folder locally:  zip -r obligation-classifier-legalbert.zip obligation-classifier-legalbert/')
        print('     Then re-upload to Drive and re-run this cell.')
else:
    print('  SKIP: obligation-classifier-legalbert not found — rule-based classifier will be used')

# Verify required directories exist
print()
ok = True
for check in ['data/02_processed/policies', 'data/02_processed/uae_ia_controls_structured.json']:
    p = pathlib.Path(REPO_DIR) / check
    if not p.exists():
        print(f'  ❌ Still missing: {check}  ← upload to Drive and re-run this cell')
        ok = False
if ok:
    print('✅ All required files in place. Proceed to next step.')

Lightning.ai: reading from persistent storage at /teamspace/studios/this_studio/compliance_pipeline_data
Copying data from Google Drive...
  ❌ MISSING (required): policies
  ❌ MISSING (required): uae_ia_controls_structured.json
  SKIP (optional, not found in Drive): golden_mapping_dataset.json
  SKIP (optional, not found in Drive): not_applicable_passages.json
  SKIP: obligation-classifier-legalbert not found — rule-based classifier will be used

✅ All required files in place. Proceed to next step.


## Step 3b — Audit policy files (deduplication check)

Detects mislabelled / duplicate files that silently hurt retrieval quality.  
If it reports missing files, upload them to Drive and re-run Step 3.

In [ ]:
import subprocess
result = subprocess.run(
    ['python3', 'scripts/deduplicate_policies.py', '--check'],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Step 4 — Install dependencies

In [ ]:
# Core pipeline dependencies (docling/flask/label-studio not needed for pipeline run)
!pip install -q \
    transformers \
    sentence-transformers \
    rank-bm25 \
    pandas \
    scikit-learn \
    numpy \
    sentencepiece

print('Install complete.')

## Step 4b — Fine-tune reranker (run once; skips automatically if already saved to Drive)

In [4]:
import os, shutil
os.chdir(REPO_DIR)

LOCAL_RERANKER = os.path.join(REPO_DIR, 'models', 'compliance-reranker')

# On Lightning.ai the model dir is already persistent — no Drive copy needed.
# On Colab we back it up to Drive so it survives runtime restarts.
if IS_LIGHTNING:
    SAVED_RERANKER = LOCAL_RERANKER   # same path — storage is persistent
else:
    SAVED_RERANKER = '/content/drive/MyDrive/compliance_pipeline_data/compliance-reranker'

def has_model_files(path):
    """True if path contains at least one model weight file anywhere inside."""
    if not os.path.exists(path):
        return False
    for root, dirs, files in os.walk(path):
        for f in files:
            if f.endswith(('.bin', '.safetensors', '.pt')):
                return True
    return False

# ── 1. Already have the model? ────────────────────────────────────────────
if has_model_files(LOCAL_RERANKER):
    print(f'Fine-tuned model already present at {LOCAL_RERANKER}')
    print('Skipping fine-tuning.')

elif not IS_LIGHTNING and has_model_files(SAVED_RERANKER):
    # Colab only: restore from Drive backup
    print('Restoring fine-tuned model from Google Drive...')
    os.makedirs(LOCAL_RERANKER, exist_ok=True)
    shutil.copytree(SAVED_RERANKER, LOCAL_RERANKER, dirs_exist_ok=True)
    print(f'  ✅ Restored: {LOCAL_RERANKER}')

# ── 2. No model found — fine-tune now ────────────────────────────────────
else:
    print('No saved model found. Running fine-tuning (~35 min on GPU)...')
    !python3 scripts/finetune_reranker.py \
        --train data/07_golden_mapping/training_data/train.json \
        --dev   data/07_golden_mapping/training_data/dev.json \
        --output models/compliance-reranker \
        --loss pairwise \
        --epochs 5 --batch-size 16

    if has_model_files(LOCAL_RERANKER):
        if not IS_LIGHTNING:
            # Colab only: back up to Drive immediately
            os.makedirs(SAVED_RERANKER, exist_ok=True)
            shutil.copytree(LOCAL_RERANKER, SAVED_RERANKER, dirs_exist_ok=True)
            print(f'\n✅ Model backed up to Drive: {SAVED_RERANKER}')
        else:
            print('\n✅ Model saved (Lightning.ai storage is persistent — no backup needed)')
    else:
        print('\n❌ No weight files found after fine-tuning. Check the output above for errors.')

# ── 3. Verify ─────────────────────────────────────────────────────────────
print('\nModel files in', LOCAL_RERANKER, ':')
file_count = 0
for root, dirs, files in os.walk(LOCAL_RERANKER):
    for f in files:
        print(f'  {os.path.relpath(os.path.join(root, f), LOCAL_RERANKER)}')
        file_count += 1
if file_count == 0:
    print('  (empty — fine-tuning may have failed)')
print(f'Done. ({file_count} files)')

No saved model found. Running fine-tuning (~35 min on GPU)...


Loading training data from data/07_golden_mapping/training_data/train.json
  Train: 2471 rows | Dev: 435 rows
  Score distribution: {'positive(1.0)': 677, 'soft_pos(0.5-0.7)': 30, 'negative(0.0)': 1764}
  Hard negatives: 1756 (duplication weight ×1.5)
  Effective training examples after hard-neg duplication: 3356
  Label smoothing ε=0.1  (1.0→0.90, 0.0→0.10)

Loading base model: BAAI/bge-reranker-base

Baseline (before fine-tuning):
  MAE=0.4012  Spearman=0.2037  BinaryAcc=0.6138

Fine-tuning for 5 epochs (batch=16, warmup=105 steps)…
  0%|                                                  | 0/1050 [00:00<?, ?it/s]Traceback (most recent call last):
  File "/teamspace/studios/this_studio/regulatory_parsing_improved/scripts/finetune_reranker.py", line 243, in <module>
    main()
  File "/teamspace/studios/this_studio/regulatory_parsing_improved/scripts/finetune_reranker.py", line 194, in main
    model.fit(
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sentenc

## Step 5 — Run the pipeline

In [5]:
import os
os.chdir(REPO_DIR)

LOCAL_RERANKER = os.path.join(REPO_DIR, 'models', 'compliance-reranker')

def has_model_files(path):
    if not os.path.exists(path): return False
    for root, dirs, files in os.walk(path):
        for f in files:
            if f.endswith(('.bin', '.safetensors', '.pt')): return True
    return False

# Using base model — best R@5 (81.5%) across all runs.
# Fine-tuning is paused; precision will be improved by Ollama LLM-as-Judge
# post-processing step instead (scripts/llm_judge.py).
reranker_model    = 'BAAI/bge-reranker-base'
threshold_full    = '0.45'
threshold_partial = '0.25'
print('Using base model: BAAI/bge-reranker-base  (R@5=81.5% — best result so far)')

# ── GPU pipeline settings ──────────────────────────────────────────────────
os.environ['CPU_MODE']          = '0'
os.environ['RERANKER_MODEL']    = reranker_model
os.environ['TOP_K_RETRIEVE']    = '50'
os.environ['TOP_K_RERANK']      = '100'
os.environ['TOP_K_PER_DOC']     = '5'
os.environ['TOP_K']             = '10'
os.environ['THRESHOLD_FULL']    = threshold_full
os.environ['THRESHOLD_PARTIAL'] = threshold_partial

print('\nPipeline settings:')
for k in ['CPU_MODE','RERANKER_MODEL','TOP_K_RETRIEVE','TOP_K_RERANK','TOP_K_PER_DOC','TOP_K',
          'THRESHOLD_FULL','THRESHOLD_PARTIAL']:
    print(f'  {k} = {os.environ[k]}')

# Run the pipeline — output files go to data/06_compliance_mappings/
!python3 quick_start_compliance.py

Using fine-tuned reranker: /teamspace/studios/this_studio/regulatory_parsing_improved/models/compliance-reranker
  Thresholds: FULL=0.38  PARTIAL=0.18 (calibrated for label-smoothed model)

Pipeline settings:
  CPU_MODE = 0
  RERANKER_MODEL = /teamspace/studios/this_studio/regulatory_parsing_improved/models/compliance-reranker
  TOP_K_RETRIEVE = 50
  TOP_K_RERANK = 100
  TOP_K_PER_DOC = 5
  TOP_K = 10
  THRESHOLD_FULL = 0.38
  THRESHOLD_PARTIAL = 0.18
UAE IA Regulation Compliance Mapping - Quick Start
The tokenizer you are loading from '/teamspace/studios/this_studio/regulatory_parsing_improved/models/compliance-reranker' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
   ✓ Reranker loaded from: /teamspace/studios/this_studio/regulatory_parsing_improved/mod

## Step 6 — Evaluate results (Precision / Recall@K / RePASs)

In [6]:
os.chdir(REPO_DIR)
!python3 scripts/evaluate_pipeline.py

  PIPELINE EVALUATION vs GOLDEN DATASET

  Golden positives (human-verified matches) : 89
  Golden negatives (human-verified non-match): 200
  Pipeline predicted positives               : 1448

  True  Positives (TP) – correct matches kept : 9
  False Positives (FP) – wrong matches surfaced: 0
  False Negatives (FN) – correct matches missed: 80

  Precision : 0.006  (0.6% of pipeline output is correct)
  Recall    : 0.101  (10.1% of human matches found)
  F1        : 0.012

  ── Missed Matches (pipeline did not surface these human-verified pairs) ──
  Control      Policy / Section
  ------------ ----------------------------------------
  M1.1.2       Security Awareness and Trainin | 9 ROLES AND RESPONSIBILITIES
  M1.3.2       Network and Communications Sec | 5.8 Confidentiality or Non-dis
  M2.1.1       Information Risk Management Po | 1 INTRODUCTION
  M2.1.1       Information Risk Management Po | 4 POLICY STATEMENT
  M2.1.1       Information Risk Management Po | 1 INTRODUCTION
  M2.1.

## Step 7 — Save results back to Google Drive

In [ ]:
import shutil, pathlib, datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
RESULTS_DIR = f'/content/drive/MyDrive/compliance_pipeline_data/results_{timestamp}'
pathlib.Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

FILES_TO_SAVE = [
    ('data/06_compliance_mappings/mappings.csv',               'mappings.csv'),
    ('data/06_compliance_mappings/mappings.json',              'mappings.json'),
    ('data/06_compliance_mappings/mappings_by_passage.json',   'mappings_by_passage.json'),
    ('data/06_compliance_mappings/compliance_report.json',     'compliance_report.json'),
    ('data/06_compliance_mappings/evaluation_report.json',     'evaluation_report.json'),
    ('data/06_compliance_mappings/retrieval_log.json',         'retrieval_log.json'),
]

for src, dst_name in FILES_TO_SAVE:
    src_path = pathlib.Path(REPO_DIR) / src
    if src_path.exists():
        shutil.copy2(src_path, f'{RESULTS_DIR}/{dst_name}')
        print(f'  ✅ saved: {dst_name}')
    else:
        print(f'  ⚠️  not found: {src}')

# Save per-policy folder
by_policy_src = pathlib.Path(REPO_DIR) / 'data/06_compliance_mappings/by_policy'
if by_policy_src.exists():
    shutil.copytree(by_policy_src, f'{RESULTS_DIR}/by_policy')
    print(f'  ✅ saved: by_policy/')

print(f'\nAll results saved to Google Drive: {RESULTS_DIR}')

## Step 8 — Push results back to GitHub

After this, just run `git pull` on your local machine and the output files appear automatically — no manual download needed.

In [ ]:
import os, subprocess
from getpass import getpass

os.chdir(REPO_DIR)

# Use the same token entered in Step 2 (re-enter if you restarted runtime)
GIT_TOKEN = getpass('GitHub token (same one used in Step 2): ').strip()
GIT_USER  = 'puneetha08nr'
GIT_EMAIL = 'puneetha@example.com'   # can be anything — just for the commit

# Configure git identity for this Colab session
subprocess.run(['git', 'config', 'user.name',  GIT_USER],  cwd=REPO_DIR)
subprocess.run(['git', 'config', 'user.email', GIT_EMAIL], cwd=REPO_DIR)

# Update remote URL to include token so push works without interactive prompt
remote_url = f'https://{GIT_TOKEN}@github.com/{GIT_USER}/regulatory_parsing_improved.git'
subprocess.run(['git', 'remote', 'set-url', 'origin', remote_url], cwd=REPO_DIR)

# Stage only the output data files (not the whole repo)
FILES_TO_COMMIT = [
    'data/06_compliance_mappings/mappings.csv',
    'data/06_compliance_mappings/mappings.json',
    'data/06_compliance_mappings/mappings_by_passage.json',
    'data/06_compliance_mappings/compliance_report.json',
    'data/06_compliance_mappings/evaluation_report.json',
    'data/06_compliance_mappings/retrieval_log.json',
    'data/06_compliance_mappings/by_policy',
]

import pathlib
for f in FILES_TO_COMMIT:
    p = pathlib.Path(REPO_DIR) / f
    if p.exists():
        subprocess.run(['git', 'add', str(p)], cwd=REPO_DIR)
        print(f'  staged: {f}')
    else:
        print(f'  skip (not found): {f}')

import datetime
run_ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
result = subprocess.run(
    ['git', 'commit', '-m', f'colab pipeline run: {run_ts}'],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(result.stdout.strip() or result.stderr.strip())

push = subprocess.run(
    ['git', 'push', 'origin', 'main'],
    cwd=REPO_DIR, capture_output=True, text=True
)
if push.returncode == 0:
    print('\n✅ Results pushed to GitHub.')
    print('   On your local machine, run:  git pull')
    print('   → data/06_compliance_mappings/ will be updated automatically.')
else:
    print('❌ Push failed:', push.stderr)
    print('   Check that your token has "repo" write access.')

## (Optional) Step 9 — Generate next annotation batch

Creates a fresh `golden_set_mapping_tasks.json` from the new pipeline output.  
Import this into Label Studio for the next annotation round.

In [ ]:
os.chdir(REPO_DIR)
!python3 create_golden_set_tasks.py \
    --candidates data/06_compliance_mappings/mappings.json \
    --output-tasks data/03_label_studio_input/golden_set_mapping_tasks.json

# Stage and push this file too
subprocess.run(['git', 'add', f'{REPO_DIR}/data/03_label_studio_input/golden_set_mapping_tasks.json'], cwd=REPO_DIR)
result = subprocess.run(['git', 'commit', '-m', 'colab: add annotation tasks for next round'],
                        cwd=REPO_DIR, capture_output=True, text=True)
subprocess.run(['git', 'push', 'origin', 'main'], cwd=REPO_DIR)
print('golden_set_mapping_tasks.json pushed to GitHub → git pull locally to get it')

## Step 10 (Optional) — Fine-tune Llama 3.2 as a compliance classifier

**When to run this:** After you have enough golden data (500+ annotated pairs).  
**What it does:** Fine-tunes `Llama-3.2-1B-Instruct` with QLoRA on your (control, passage, label) pairs.  
**Output:** `models/llama-compliance/` — LoRA adapters; the model classifies *and explains* matches.  
**Time:** ~45–60 min on T4 GPU (1B model, 3 epochs).  

> **Recommended order:** Fine-tune the cross-encoder first (`finetune_reranker.py`) for the pipeline.  
> Use Llama for generating *explanations* of why a match is correct — useful in audit reports.